# 06. Preparación de Datos para Deep Learning

Este notebook transforma los artefactos generados en datasets listos para un modelo neural de ranking o clasificación usuario-item.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

import json
import numpy as np

from src.utils.recsys_utils import ensure_dir

processed_dir = ensure_dir(ROOT / 'data' / 'processed')

train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
valid = pd.read_parquet(processed_dir / 'interactions_valid.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')
item_features = pd.read_parquet(processed_dir / 'item_features.parquet')
user_index = pd.read_parquet(processed_dir / 'user_index.parquet')
item_index = pd.read_parquet(processed_dir / 'item_index.parquet')


## Seleccionar positivos y generar negativos

Usamos `rating >= 4` como señal positiva y muestreo negativo aleatorio por usuario.


In [ ]:
positive_train = train[train['rating'] >= 4.0][['userId', 'movieId', 'user_idx', 'item_idx']].drop_duplicates().copy()
all_items = item_index['movieId'].tolist()
seen_per_user = train.groupby('userId')['movieId'].apply(set).to_dict()

rng = np.random.default_rng(42)
negative_rows = []
for row in positive_train.itertuples(index=False):
    seen = seen_per_user.get(int(row.userId), set())
    candidate = int(rng.choice(all_items))
    attempts = 0
    while candidate in seen and attempts < 20:
        candidate = int(rng.choice(all_items))
        attempts += 1
    negative_rows.append({'userId': int(row.userId), 'movieId': candidate, 'label': 0})

positive_rows = positive_train[['userId', 'movieId']].assign(label=1)
train_dl = pd.concat([positive_rows, pd.DataFrame(negative_rows)], ignore_index=True)
train_dl = train_dl.merge(user_index, on='userId', how='left').merge(item_index, on='movieId', how='left')
train_dl = train_dl.merge(item_features, on='movieId', how='left')
display(train_dl.head())
print('Dataset de training DL:', train_dl.shape)


## Preparar validation/test y mapas de embeddings

Los splits de validación y test quedan en el mismo formato supervisado, listos para PyTorch o TensorFlow.


In [ ]:
def build_eval_frame(frame):
    out = frame[['userId', 'movieId', 'rating', 'user_idx', 'item_idx']].copy()
    out['label'] = (out['rating'] >= 4.0).astype(int)
    out = out.merge(item_features, on='movieId', how='left')
    return out

valid_dl = build_eval_frame(valid)
test_dl = build_eval_frame(test)

embedding_maps = {
    'n_users': int(user_index['user_idx'].nunique()),
    'n_items': int(item_index['item_idx'].nunique()),
    'item_feature_columns': [col for col in item_features.columns if col not in {'movieId', 'title', 'clean_title'}],
}

train_dl.to_parquet(processed_dir / 'dl_train.parquet', index=False)
valid_dl.to_parquet(processed_dir / 'dl_valid.parquet', index=False)
test_dl.to_parquet(processed_dir / 'dl_test.parquet', index=False)
(processed_dir / 'embedding_maps.json').write_text(json.dumps(embedding_maps, indent=2, ensure_ascii=True))

pd.DataFrame([embedding_maps]).to_csv(processed_dir / 'dl_metadata.csv', index=False)
display(pd.DataFrame([embedding_maps]))
